# Weight Initialization Techniques
## DLMDSDL01 Deep Learning - Prof. Dr. Heinke Hihn
# IU International University of Applied Sciences

### Learning Objectives
1. Understand why weight initialization matters for training stability
2. Implement and compare different initialization strategies
3. Visualize the effect of initialization on activation distributions
4. Recognize vanishing and exploding gradient problems

See Goodfellow et al. (2016) Deep Learning, Section 8.4:
           Parameter Initialization Strategies

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import pickle
import time

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("Setup complete!")
print(f"PyTorch version: {torch.__version__}")

As stated in the course book:
"At the most basic level, training a deep neural network means initializing
the network weights randomly and then iteratively updating them via
backpropagation and gradient descent until a local minimum of the loss
function is reached."

The key insight: Poor initialization leads to:
- Vanishing gradients (gradients → 0)
- Exploding gradients (gradients → ∞)
- Symmetry problem (all neurons learn the same thing)

In [ ]:
# Let's demonstrate the problem with a simple forward pass simulation
def simulate_forward_pass(weight_scale, num_layers=6, layer_size=256, activation='tanh'):
    """
    Simulate forward pass through multiple layers and track activation statistics.
    """
    activations = []
    x = np.random.randn(layer_size)  # Random input

    for i in range(num_layers):
        # Initialize weights with given scale
        W = weight_scale * np.random.randn(layer_size, layer_size)
        x = np.dot(x, W)

        # Apply activation function
        if activation == 'tanh':
            x = np.tanh(x)
        elif activation == 'relu':
            x = np.maximum(0, x)
        elif activation == 'sigmoid':
            x = 1 / (1 + np.exp(-np.clip(x, -500, 500)))

        activations.append(x.copy())

    return activations


# Exercise 1: Observe the Effect of Different Weight Scales

In [ ]:
print("\n" + "="*60)
print("EXERCISE 1: Effect of Weight Scale on Activations")
print("="*60)

weight_scales = [0.01, 0.5, 1.0]
fig, axes = plt.subplots(len(weight_scales), 6, figsize=(18, 3*len(weight_scales)))

for row, scale in enumerate(weight_scales):
    activations = simulate_forward_pass(scale, num_layers=6, activation='tanh')

    for col, act in enumerate(activations):
        axes[row, col].hist(act, bins=50, density=True, alpha=0.7)
        axes[row, col].set_xlim(-1.2, 1.2)
        axes[row, col].set_title(f'Scale={scale}, Layer {col+1}\nμ={act.mean():.3f}, σ={act.std():.3f}')

plt.tight_layout()
plt.suptitle('Activation Distributions Across Layers (tanh activation)', y=1.02, fontsize=14)
plt.show()

# TODO: Answer these questions:
# Q1: What happens with scale=0.01? Why is this problematic?
# Q2: What happens with scale=1.0? Why is this problematic?
# Q3: Which scale seems to maintain stable activations?

# Naive Initializaiton Approaches

From the course materials (Session 3):

APPROACH 1: Zero Initialization
- Method: Set all weights to 0
- Result: FAILURE - Symmetry is not broken. All neurons compute identical outputs.

APPROACH 2: Small Random Numbers (Standard Normal)
- Method: Draw from N(0, σ²) with small σ
- Problem: For deep networks, activations collapse to zero

In [ ]:
# Demonstrate zero initialization problem
class ZeroInitNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

        # Zero initialization (BAD!)
        nn.init.zeros_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)
        nn.init.zeros_(self.fc3.weight)
        nn.init.zeros_(self.fc3.bias)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# Test zero initialization
zero_net = ZeroInitNetwork()
test_input = torch.randn(1, 784)
output = zero_net(test_input)
print("Zero Initialization Test:")
print(f"  Output: {output.detach().numpy()}")
print(f"  All outputs identical? {len(torch.unique(output)) == 1}")
print("  This network cannot learn - all gradients will be identical!")

# Xavier/Glorot Initialization

From Glorot & Bengio (2010):

Xavier initialization maintains variance across layers by scaling weights:

For UNIFORM distribution:
$W \sim U[-\sqrt{6/(fan_{in} + fan_{out})}, \sqrt{6/(fan_{in} + fan_{out})}]$

For NORMAL distribution:
$W \sim N(0, \sqrt{2/(fan_in + fan_out)})$

Best suited for: sigmoid, tanh activations

In [ ]:
def xavier_uniform(fan_in, fan_out):
    """Implement Xavier uniform initialization manually."""
    # TODO: implement the initialization technique mentioned above
    pass

def xavier_normal(fan_in, fan_out):
    """Implement Xavier normal initialization manually."""
    # TODO: implement the initialization technique mentioned above
    pass

# Verify Xavier maintains variance
print("\Xavier Initialization Variance Check")
fan_in, fan_out = 256, 256

W_xavier = xavier_normal(fan_in, fan_out)
print(f"Manual Xavier Normal - Mean: {W_xavier.mean():.6f}, Std: {W_xavier.std():.6f}")
print(f"Expected Std: {np.sqrt(2.0 / (fan_in + fan_out)):.6f}")

W_pytorch = torch.empty(fan_in, fan_out)
nn.init.xavier_normal_(W_pytorch)
print(f"PyTorch Xavier Normal - Mean: {W_pytorch.mean():.6f}, Std: {W_pytorch.std():.6f}")

<details>
<summary>Click here for the solution </summary>

Uniform
```python
limit = np.sqrt(6.0 / (fan_in + fan_out))
return np.random.uniform(-limit, limit, size=(fan_in, fan_out))
```
Normal
```python
std = np.sqrt(2.0 / (fan_in + fan_out))
return np.random.normal(0, std, size=(fan_in, fan_out))
```

# He/Kaiming Initialization

From He et al. (2015) - "Delving Deep into Rectifiers":

For ReLU activations, Xavier doesn't account for the fact that
ReLU zeros out ~50% of activations. He initialization corrects this:

For NORMAL distribution:
$W \sim N(0, \sqrt{2/fan_{in}})$

Best suited for: ReLU, LeakyReLU, PReLU activations

In [ ]:
def he_normal(fan_in, fan_out):
    """Implement He/Kaiming normal initialization manually."""
    # TODO: implement the initialization
    pass

# Compare Xavier vs He for ReLU networks
print("\nComparing Xavier vs He for ReLU networks:")

def simulate_with_init(init_func, num_layers=6, layer_size=256):
    """Simulate forward pass with specified initialization."""
    x = np.random.randn(layer_size)
    stds = []

    for i in range(num_layers):
        W = init_func(layer_size, layer_size)
        x = np.dot(x, W)
        x = np.maximum(0, x)  # ReLU
        stds.append(x.std())

    return stds

xavier_stds = simulate_with_init(xavier_normal)
he_stds = simulate_with_init(he_normal)

plt.figure(figsize=(10, 5))
plt.plot(range(1, 7), xavier_stds, 'b-o', label='Xavier Normal', linewidth=2)
plt.plot(range(1, 7), he_stds, 'r-o', label='He Normal', linewidth=2)
plt.xlabel('Layer', fontsize=12)
plt.ylabel('Activation Std Dev', fontsize=12)
plt.title('Xavier vs He Initialization with ReLU Activation', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Question:
# Which appraoch is more stable with ReLU?

<details>
<summary>Click here for the solution</summary>

```python
std = np.sqrt(2.0 / fan_in)
return np.random.normal(0, std, size=(fan_in, fan_out))
```

# PyTorch Initialization Methods

PyTorch does NOT use optimal initialization by default!
- Conv2d and Linear use kaiming_uniform_ with a=sqrt(5) by default
- This is for backward compatibility, not optimal performance

BEST PRACTICE: Always explicitly initialize your weights!

# Hands-On Challange: Train MNIST with different initialization techniques

Create the network:

In [ ]:
class WellInitializedNetwork(nn.Module):
    def __init__(self, init_method='kaiming'):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

        self._initialize_weights(init_method)

    def _initialize_weights(self, method):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if method == 'xavier_uniform':
                    nn.init.xavier_uniform_(m.weight)
                elif method == 'xavier_normal':
                    nn.init.xavier_normal_(m.weight)
                elif method == 'kaiming_uniform':
                    nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                elif method == 'kaiming_normal':
                    nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                elif method == 'zeros':
                    nn.init.zeros_(m.weight)

                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

In [ ]:
# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

def train_with_init(init_method, epochs=3):
    """Train network and return loss history."""
    model = WellInitializedNetwork(init_method=init_method)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    losses = []
    for epoch in range(epochs):
        epoch_loss = 0
        for batch_idx, (data, target) in enumerate(train_loader):
            data = data.view(-1, 784)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

            if batch_idx >= 100:  # Limit for speed
                break
        losses.append(epoch_loss / 100)
    return losses

Compare

In [ ]:
methods = ['xavier_uniform', 'xavier_normal', 'kaiming_uniform', 'kaiming_normal']

for method in methods:
    net = WellInitializedNetwork(init_method=method)
    w1 = net.fc1.weight.data
    print(f"{method:20s} - Mean: {w1.mean():8.5f}, Std: {w1.std():.5f}")

In [ ]:
def analyze_gradient_flow(model, input_data, target):
    """Analyze gradient magnitudes through the network."""
    model.zero_grad()
    output = model(input_data)
    loss = nn.functional.cross_entropy(output, target)
    loss.backward()

    gradients = []
    for name, param in model.named_parameters():
        if 'weight' in name and param.grad is not None:
            gradients.append((name, param.grad.abs().mean().item()))

    return gradients

# Compare gradient flow for different initializations
test_input = torch.randn(32, 784)
test_target = torch.randint(0, 10, (32,))

fig, axes = plt.subplots(1, 5, figsize=(14, 5))

for idx, method in enumerate(['zeros', 'xavier_uniform', 'xavier_normal', 'kaiming_uniform', 'kaiming_normal']):
    net = WellInitializedNetwork(init_method=method)
    grads = analyze_gradient_flow(net, test_input, test_target)

    names = [g[0].replace('.weight', '') for g in grads]
    values = [g[1] for g in grads]

    axes[idx].bar(names, values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
    axes[idx].set_title(f'{method}', fontsize=12)
    axes[idx].set_ylabel('Mean Absolute Gradient')
    axes[idx].set_yscale('log')

plt.tight_layout()
plt.show()


Train a network on MNIST with different initializations
and compare the training dynamics.

TODO:
1. Complete the training loop below
2. Compare loss curves for different initialization methods
3. Answer: Which initialization leads to fastest convergence?

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Load MNIST (subset for speed)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

def train_with_init(init_method, epochs=10):
    """Train network and return loss history."""
    model = WellInitializedNetwork(init_method=init_method)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    losses = []

    for epoch in range(epochs):
        epoch_loss = 0.0
        num_batches = 0

        for batch_idx, (data, target) in enumerate(train_loader):
            # Flatten MNIST images from (batch, 1, 28, 28) to (batch, 784)
            data = data.view(-1, 784)

            # Forward pass


            # Backward pass


            epoch_loss += loss.item()
            num_batches += 1


        avg_loss = epoch_loss / num_batches
        losses.append(avg_loss)
        print(f"  Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    return losses

# Compare different initialization methods
print("Training with different initializations...")
results = {}
for method in ['zeros', 'xavier_normal', 'kaiming_normal']:
    print(f"\n{method}:")
    results[method] = train_with_init(method, epochs=3)

# Plot comparison
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
for method, losses in results.items():
    plt.plot(losses, '-o', label=method, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss by Initialization Method')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

<details>
<summary>click here for the solution </summary>
```python
# Forward pass
optimizer.zero_grad()
output = model(data)
loss = criterion(output, target)

# Backward pass
loss.backward()
optimizer.step()

epoch_loss += loss.item()
num_batches += 1